# MSE Budget Analysis — ORCESTRA Dropsonde Circles

This notebook computes the **column Moist Static Energy (MSE) budget** from the BEACH Level-4 dropsonde dataset using two methods:

| Method | Equation | Source |
|--------|----------|--------|
| **Advective Form** | $\frac{\partial \langle h \rangle}{\partial t} + \langle \mathbf{v} \cdot \nabla h \rangle + \langle \omega \frac{\partial h}{\partial p} \rangle = F$ | Circle-mean fields |
| **Flux Form (Line Integral)** | $\frac{\partial \langle h \rangle}{\partial t} + \langle \nabla \cdot (\mathbf{v}h) \rangle = F$ | Individual sondes via divergence theorem |

See `notes/MSE_budget_methods.md` for full derivations.

In [ ]:
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Add project scripts to path
sys.path.insert(0, "/home/565/zr7147/Proj/scripts")
import mse_budget as mseb

# Reload if you edit the script
import importlib
importlib.reload(mseb)


In [ ]:

ZARR_PATH = "/g/data/k10/zr7147/ORCESTRA_dropsondes_categorized.zarr"
ds = mseb.load_dataset(ZARR_PATH)
print(f"Loaded: {ds.sizes['circle']} circles, {ds.sizes['sonde']} sondes, {ds.sizes['altitude']} levels")

## 1. Choose Method

Set `METHOD` to one of:
- `"advective"` — uses circle-mean fields (fast, separates horizontal / vertical advection)
- `"flux"` — uses individual sonde data via line integral (slower, single total-export term)
- `"both"` — runs both for cross-validation

In [ ]:
# ---- SELECT METHOD HERE ----
METHOD = "both"  # options: "advective", "flux", "both"

print(f"Computing MSE budget using: {METHOD}")
budget = mseb.compute_budget(ds, methods=METHOD)
print("Done.")
print()
print("Available budget variables:")
for v in sorted(budget.data_vars):
    if budget[v].dims == ('circle',):
        print(f"  {v}")

## 2. Budget Summary by Category

In [ ]:
cat = budget["category_evolutionary"].values
angle = budget["top_heaviness_angle"].values

cats_to_show = ["Top-Heavy", "Bottom-Heavy", "Inactive"]

print(f"{'Category':<30} {'N':>4}  ", end="")
if "vert_adv" in budget:
    print(f"{'<ω∂h/∂p>':>12} {'<v·∇h>':>12} {'GMS':>8}", end="")
if "flux_div_mse" in budget:
    print(f"{'<∇·(vh)>':>12} {'GMS_flux':>10}", end="")
print()
print("-" * 100)

for label in cats_to_show:
    mask = np.array([label in str(c) for c in cat])
    n = mask.sum()
    print(f"{label:<30} {n:>4}  ", end="")
    if "vert_adv" in budget:
        va  = np.nanmean(budget["vert_adv"].values[mask])
        ha  = np.nanmean(budget["horiz_adv"].values[mask])
        gms = np.nanmean(budget["gms"].values[mask])
        print(f"{va:>+12.1f} {ha:>+12.1f} {gms:>8.3f}", end="")
    if "flux_div_mse" in budget:
        fd  = np.nanmean(budget["flux_div_mse"].values[mask])
        gf  = np.nanmean(budget["gms_flux"].values[mask])
        print(f"{fd:>+12.1f} {gf:>10.3f}", end="")
    print()

## 3. Omega Profile Shape vs Energy Export Efficiency

This is the key plot: **Singh & Li angle** (profile shape) vs **GMS** (energy export efficiency).

In [ ]:
fig, axes = plt.subplots(1, 2 if "gms_flux" in budget else 1,
                         figsize=(14, 5) if "gms_flux" in budget else (7, 5),
                         sharey=True)
if not hasattr(axes, '__len__'):
    axes = [axes]

color_map = {
    "Top-Heavy": "red",
    "Bottom-Heavy": "blue",
    "Inactive": "gray",
}

def plot_angle_vs_gms(ax, gms_var, title):
    gms_vals = budget[gms_var].values
    for label, color in color_map.items():
        mask = (np.array([label in str(c) for c in cat])
                & ~np.isnan(angle) & ~np.isnan(gms_vals))
        ax.scatter(angle[mask], gms_vals[mask], c=color, label=label,
                   s=50, alpha=0.7, edgecolors="k", linewidths=0.3)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.axhline(1, color="gray", ls=":", lw=0.6, alpha=0.5)
    ax.axvline(45, color="gray", ls=":", lw=0.8)
    ax.set_xlabel("Top-Heaviness Angle (°)", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)

# Advective GMS
if "gms" in budget:
    plot_angle_vs_gms(axes[0], "gms", "Advective Form: GMS")
    axes[0].set_ylabel("Normalised GMS  $\\tilde{M}$", fontsize=12)

# Flux GMS
if "gms_flux" in budget and len(axes) > 1:
    plot_angle_vs_gms(axes[1], "gms_flux", "Flux Form (Line Integral): GMS")

fig.suptitle("Vertical Motion Profile Shape vs Energy Export Efficiency",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Cross-Validation: Advective vs Flux Form

If both methods were run, the **total advective export** $\langle \mathbf{v} \cdot \nabla h \rangle + \langle \omega \partial h / \partial p \rangle$ should approximately equal the **flux divergence** $\langle \nabla \cdot (\mathbf{v}h) \rangle$.  The scatter below checks this closure.

In [ ]:
if "total_adv" in budget and "flux_div_mse" in budget:
    total_adv = budget["total_adv"].values
    flux_div  = budget["flux_div_mse"].values

    valid = ~np.isnan(total_adv) & ~np.isnan(flux_div)

    fig, ax = plt.subplots(figsize=(6, 6))
    for label, color in color_map.items():
        mask = valid & np.array([label in str(c) for c in cat])
        ax.scatter(total_adv[mask], flux_div[mask], c=color, label=label,
                   s=50, alpha=0.7, edgecolors="k", linewidths=0.3)

    lims = [min(np.nanmin(total_adv[valid]), np.nanmin(flux_div[valid])),
            max(np.nanmax(total_adv[valid]), np.nanmax(flux_div[valid]))]
    ax.plot(lims, lims, "k--", lw=1, label="1:1 line")
    ax.set_xlabel("Advective: $\\langle \\mathbf{v}{\\cdot}\\nabla h \\rangle + \\langle \\omega \\partial h/\\partial p \\rangle$  (W m$^{-2}$)", fontsize=10)
    ax.set_ylabel("Flux: $\\langle \\nabla{\\cdot}(\\mathbf{v}h) \\rangle$  (W m$^{-2}$)", fontsize=10)
    ax.set_title("Cross-validation: Advective vs Flux Form", fontsize=12)
    ax.set_aspect("equal")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    # Correlation
    r = np.corrcoef(total_adv[valid], flux_div[valid])[0, 1]
    rmse = np.sqrt(np.nanmean((total_adv[valid] - flux_div[valid])**2))
    print(f"Correlation: r = {r:.3f}")
    print(f"RMSE:        {rmse:.1f} W m⁻²")
else:
    print("Run with METHOD='both' to see cross-validation.")

## 5. Budget Term Profiles (Advective)

Composite vertical profiles of each budget term, grouped by omega category.

In [ ]:
if "h_profile" in budget:
    omega_vals = budget["omega"].values   # (circle, altitude)
    p_mean = ds["p_mean"].values          # Pa
    h_prof = budget["h_profile"].values

    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

    for label, color in [("Top-Heavy", "red"), ("Bottom-Heavy", "blue")]:
        mask = np.array([label in str(c) for c in cat])
        if mask.sum() == 0:
            continue

        # Omega composite
        om_comp = np.nanmean(omega_vals[mask], axis=0)
        p_comp  = np.nanmean(p_mean[mask], axis=0) / 100  # hPa for plotting

        # MSE composite
        h_comp = np.nanmean(h_prof[mask], axis=0) / 1000  # kJ/kg

        # Vertical advection profile: ω·∂h/∂p at each level
        vadv_profiles = []
        for ci in np.where(mask)[0]:
            dhdp = np.full_like(h_prof[ci], np.nan)
            v_ok = ~np.isnan(h_prof[ci]) & ~np.isnan(p_mean[ci])
            if v_ok.sum() > 2:
                dhdp[v_ok] = np.gradient(h_prof[ci, v_ok], p_mean[ci, v_ok])
            vadv_profiles.append(omega_vals[ci] * dhdp)
        vadv_comp = np.nanmean(vadv_profiles, axis=0)

        axes[0].plot(om_comp, p_comp, color=color, lw=2, label=label)
        axes[1].plot(h_comp, p_comp, color=color, lw=2, label=label)
        axes[2].plot(vadv_comp, p_comp, color=color, lw=2, label=label)

    for ax in axes:
        ax.invert_yaxis()
        ax.set_ylabel("Pressure (hPa)")
        ax.legend(fontsize=9)
        ax.axvline(0, color="gray", ls="--", lw=0.5)

    axes[0].set_xlabel("$\\omega$ (Pa s$^{-1}$)")
    axes[0].set_title("Omega Profile")
    axes[1].set_xlabel("MSE $h$ (kJ kg$^{-1}$)")
    axes[1].set_title("MSE Profile")
    axes[2].set_xlabel("$\\omega \\partial h / \\partial p$  (J kg$^{-1}$ s$^{-1}$ Pa$^{-1}$)")
    axes[2].set_title("Vertical MSE Advection")

    fig.suptitle("Composite Profiles by Category", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Run with METHOD='advective' or 'both' to see profiles.")

## 6. GMS Distribution by Category

In [ ]:
gms_var = "gms" if "gms" in budget else "gms_flux"
gms_vals = budget[gms_var].values

fig, ax = plt.subplots(figsize=(8, 5))

for label, color in [("Top-Heavy", "red"), ("Bottom-Heavy", "blue")]:
    mask = np.array([label in str(c) for c in cat]) & ~np.isnan(gms_vals)
    ax.hist(gms_vals[mask], bins=15, alpha=0.5, color=color, label=label, edgecolor="k")

ax.axvline(0, color="gray", ls="--", lw=1)
ax.set_xlabel("Normalised GMS $\\tilde{M}$", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"GMS Distribution by Category ({gms_var})", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
output_path = "/g/data/k10/zr7147/mse_budget_results.nc"

# Drop non-serializable string variables for netCDF
save_vars = [v for v in budget.data_vars
             if budget[v].dtype.kind in ('f', 'i')]
budget[save_vars].to_netcdf(output_path)
print(f"Saved numeric budget results to {output_path}")